# Prediksi Ketidakpatuhan Pasien dengan Deep Learning, Imbalanced Learning, dan SHAP

Notebook ini digunakan untuk eksperimen tesis:
**Pengembangan Model Deep Learning untuk Prediksi Ketidakpatuhan Pasien pada Dataset Tidak Seimbang dengan Pendekatan Imbalanced Learning dan Explainable AI**.

Tujuan utama:
- membangun pipeline eksperimen yang reproducible dan bebas leakage
- membandingkan model deep learning tabular dengan beberapa strategi imbalance handling
- mengevaluasi metrik yang relevan untuk data tidak seimbang
- menginterpretasikan model terbaik menggunakan SHAP

Catatan:
- Eksperimen ini menggunakan **maksimal 10.000 baris data** untuk efisiensi komputasi awal.
- SHAP dilakukan **setelah** model terbaik dipilih dari cross-validation dan evaluasi final.


In [2]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    auc,
)
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import shap
import joblib

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Konfigurasi eksperimen
DATASET_MODE = "combined"      # opsi: "diabetes", "htn", "combined"
MAX_ROWS = 10_000
RANDOM_STATE = 42
N_SPLITS = 5
TEST_SIZE = 0.2
ADHERENCE_THRESHOLD = 9.0       # threshold operasional (bukan klaim klinis)
TARGET_COLUMN = "ADHERENCE"
POSITIVE_IF_LOWER = True        # y=1 jika ADHERENCE < threshold
GROUP_COLUMN = "MEMBER"        # split dan CV berbasis pasien

DATA_DIR = Path("dataset")
OUTPUT_DIR = Path("outputs")
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
SHAP_DIR = OUTPUT_DIR / "shap"
MODELS_DIR = OUTPUT_DIR / "models"

for p in [OUTPUT_DIR, METRICS_DIR, FIGURES_DIR, SHAP_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Config loaded")
print({
    "DATASET_MODE": DATASET_MODE,
    "MAX_ROWS": MAX_ROWS,
    "RANDOM_STATE": RANDOM_STATE,
    "N_SPLITS": N_SPLITS,
    "TEST_SIZE": TEST_SIZE,
    "ADHERENCE_THRESHOLD": ADHERENCE_THRESHOLD,
    "GROUP_COLUMN": GROUP_COLUMN,
})


In [ ]:
# Load dataset
DIAB_PATH = DATA_DIR / "Diabetes Adherence Data.csv"
HTN_PATH = DATA_DIR / "HTN Adherence Data.csv"

if not DIAB_PATH.exists() or not HTN_PATH.exists():
    raise FileNotFoundError(f"CSV tidak ditemukan di folder {DATA_DIR.resolve()}")

df_diabetes_raw = pd.read_csv(DIAB_PATH, low_memory=False)
df_htn_raw = pd.read_csv(HTN_PATH, low_memory=False)

print("Diabetes shape:", df_diabetes_raw.shape)
print("HTN shape:", df_htn_raw.shape)

print("Preview Diabetes")
display(df_diabetes_raw.head())
print("Preview HTN")
display(df_htn_raw.head())

print("Kolom Diabetes:")
print(df_diabetes_raw.columns.tolist())

print("Dtypes Diabetes (top):")
display(df_diabetes_raw.dtypes.to_frame("dtype").head(20))

print("Missing values Diabetes (top):")
display(df_diabetes_raw.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_count"))

print("Dtypes HTN (top):")
display(df_htn_raw.dtypes.to_frame("dtype").head(20))

print("Missing values HTN (top):")
display(df_htn_raw.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_count"))


In [ ]:
# Pilih dataset sesuai mode
if DATASET_MODE == "diabetes":
    df_selected = df_diabetes_raw.copy()
elif DATASET_MODE == "htn":
    df_selected = df_htn_raw.copy()
elif DATASET_MODE == "combined":
    d1 = df_diabetes_raw.copy()
    d2 = df_htn_raw.copy()
    d1["disease_type"] = "diabetes"
    d2["disease_type"] = "htn"
    df_selected = pd.concat([d1, d2], ignore_index=True)
else:
    raise ValueError("DATASET_MODE harus diabetes/htn/combined")

print("Selected mode:", DATASET_MODE)
print("Selected dataset shape:", df_selected.shape)


In [ ]:
# EDA awal
print("Info dataset")
print(df_selected.info())

print("Statistik deskriptif numerik")
display(df_selected.describe(include=[np.number]).T)

print("Jumlah unique tiap kolom (top 30)")
unique_counts = df_selected.nunique(dropna=False).sort_values(ascending=False)
display(unique_counts.head(30).to_frame("n_unique"))

print("Missing percentage tiap kolom (top 30)")
missing_pct = (df_selected.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.head(30).to_frame("missing_pct"))

if TARGET_COLUMN not in df_selected.columns:
    raise KeyError(f"Kolom target {TARGET_COLUMN} tidak ditemukan")

adh = pd.to_numeric(df_selected[TARGET_COLUMN], errors="coerce")
print("Distribusi ADHERENCE (deskriptif)")
print(adh.describe())
print("Value counts ADHERENCE")
display(adh.value_counts(dropna=False).sort_index().to_frame("count"))

plt.figure(figsize=(10, 4))
sns.histplot(adh.dropna(), bins=30, kde=True)
plt.title("Distribusi ADHERENCE")
plt.xlabel("ADHERENCE")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## Sampling Maksimal 10.000 Data

Pada tahap ini, sampling dilakukan untuk efisiensi komputasi dan iterasi eksperimen awal.
- Jika data > 10.000 baris, data disampling menjadi 10.000 baris.
- Karena target biner belum dibentuk pada tahap ini, sampling dilakukan secara acak terkontrol (`random_state`).
- Setelah target biner terbentuk, split dan cross-validation akan menggunakan stratifikasi untuk menjaga distribusi kelas.


In [ ]:
# Sampling maksimal 10.000 baris
n_before = len(df_selected)
adh_before = pd.to_numeric(df_selected[TARGET_COLUMN], errors="coerce")

if n_before > MAX_ROWS:
    df_sampled = df_selected.sample(n=MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    df_sampled = df_selected.copy().reset_index(drop=True)

n_after = len(df_sampled)
adh_after = pd.to_numeric(df_sampled[TARGET_COLUMN], errors="coerce")

print(f"Rows before sampling: {n_before:,}")
print(f"Rows after sampling : {n_after:,}")

print("Distribusi ADHERENCE sebelum sampling (top 15)")
display(adh_before.value_counts(dropna=False).sort_index().head(15).to_frame("count"))

print("Distribusi ADHERENCE sesudah sampling (top 15)")
display(adh_after.value_counts(dropna=False).sort_index().head(15).to_frame("count"))

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
sns.histplot(adh_before.dropna(), bins=30, kde=True)
plt.title("ADHERENCE Sebelum Sampling")
plt.subplot(1, 2, 2)
sns.histplot(adh_after.dropna(), bins=30, kde=True)
plt.title("ADHERENCE Sesudah Sampling")
plt.tight_layout()
plt.show()


## Audit Kolom dan Potensi Data Leakage

Tahap ini menyusun audit kolom (tipe data, missing, unique) dan rekomendasi:
- `keep`: aman dipertahankan
- `drop`: identifier, administratif, duplikasi, atau sangat berisiko leakage
- `transform`: kolom tanggal/kolom yang perlu rekayasa fitur

Kolom identifier dan administratif berisiko leakage di-drop dari pipeline modeling.


In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip().upper().replace(" ", "_") for c in out.columns]
    return out


df_work = normalize_columns(df_sampled)
TARGET_COLUMN_STD = TARGET_COLUMN.strip().upper().replace(" ", "_")
GROUP_COLUMN_STD = GROUP_COLUMN.strip().upper().replace(" ", "_")

if GROUP_COLUMN_STD not in df_work.columns:
    raise KeyError(f"Kolom group {GROUP_COLUMN_STD} tidak ditemukan")

# MEMBER sengaja tidak di-drop karena dipakai untuk split/CV berbasis pasien
explicit_drop_cols = {
    "INO", "CLAIM_NO", "CLAIM_LINE_NO", "INV_REF", "AUTH_NO",
    "ASSESSOR_NAME", "DUPLICATE_CLAIM", "DUPLICATE_CLAIM_LINE",
    "PZTIQNT_NZXQ", "PAY_TO"
}

date_candidate_cols = {"SERVICE_DATE", "ASSESS_DATE", "DATE_RECEIVED", "BIRTHDATE"}

missing_ratio = df_work.isna().mean()
all_null_cols = set(missing_ratio[missing_ratio >= 1.0].index)
high_missing_cols = set(missing_ratio[missing_ratio >= 0.95].index)

recommendations = []
for col in df_work.columns:
    miss = int(df_work[col].isna().sum())
    uniq = int(df_work[col].nunique(dropna=False))
    dtype = str(df_work[col].dtype)

    if col in explicit_drop_cols:
        rec, reason = "drop", "Identifier/administrative leakage risk"
    elif col in all_null_cols:
        rec, reason = "drop", "Completely empty"
    elif col in high_missing_cols:
        rec, reason = "drop", "Very high missing ratio (>=95%)"
    elif col in date_candidate_cols:
        rec, reason = "transform", "Date feature engineering candidate"
    elif col == GROUP_COLUMN_STD:
        rec, reason = "keep", "Group identifier for patient-based split/CV"
    elif col == TARGET_COLUMN_STD:
        rec, reason = "keep", "Original adherence score for target construction"
    else:
        rec, reason = "keep", "Potential predictor"

    recommendations.append({
        "column": col,
        "dtype": dtype,
        "missing_count": miss,
        "missing_pct": round(100 * miss / len(df_work), 2),
        "n_unique": uniq,
        "recommendation": rec,
        "reason": reason,
    })

audit_df = pd.DataFrame(recommendations).sort_values(["recommendation", "missing_pct"], ascending=[True, False])
display(audit_df)

cols_to_drop = sorted((explicit_drop_cols | all_null_cols | high_missing_cols) - {TARGET_COLUMN_STD, GROUP_COLUMN_STD})
print("Columns to drop:", cols_to_drop)

df_work = df_work.drop(columns=[c for c in cols_to_drop if c in df_work.columns], errors="ignore")
print("Shape after leakage-oriented drop:", df_work.shape)


In [ ]:
# Parsing tanggal dan feature engineering tanggal
for c in ["SERVICE_DATE", "ASSESS_DATE", "DATE_RECEIVED", "BIRTHDATE"]:
    if c in df_work.columns:
        df_work[c] = pd.to_datetime(df_work[c], errors="coerce", dayfirst=True)

if "SERVICE_DATE" in df_work.columns:
    df_work["SERVICE_YEAR"] = df_work["SERVICE_DATE"].dt.year
    df_work["SERVICE_MONTH"] = df_work["SERVICE_DATE"].dt.month
    df_work["SERVICE_DAYOFWEEK"] = df_work["SERVICE_DATE"].dt.dayofweek

if "BIRTHDATE" in df_work.columns and "SERVICE_DATE" in df_work.columns:
    df_work["AGE_FROM_BIRTHDATE"] = ((df_work["SERVICE_DATE"] - df_work["BIRTHDATE"]).dt.days / 365.25)

# Kolom tanggal mentah di-drop agar lebih aman dan tidak ambigu
raw_date_cols = [c for c in ["SERVICE_DATE", "ASSESS_DATE", "DATE_RECEIVED", "BIRTHDATE"] if c in df_work.columns]
df_work = df_work.drop(columns=raw_date_cols, errors="ignore")

print("Raw date columns dropped:", raw_date_cols)
print("Current shape:", df_work.shape)

new_date_features = [c for c in ["SERVICE_YEAR", "SERVICE_MONTH", "SERVICE_DAYOFWEEK", "AGE_FROM_BIRTHDATE"] if c in df_work.columns]
print("Date-derived features:", new_date_features)
display(df_work[new_date_features].head() if new_date_features else pd.DataFrame())


## Target Construction (Operasional)

Target biner dibentuk dari skor `ADHERENCE` menggunakan threshold operasional untuk kebutuhan eksperimen.
Threshold ini **bukan klaim klinis** tanpa dokumentasi resmi eksternal.


In [ ]:
def create_binary_target(df: pd.DataFrame, target_col: str, threshold: float, positive_if_lower: bool = True) -> pd.DataFrame:
    out = df.copy()
    y_raw = pd.to_numeric(out[target_col], errors="coerce")
    out = out[y_raw.notna()].copy()
    y_raw = y_raw[y_raw.notna()]

    if positive_if_lower:
        out["TARGET_BINARY"] = (y_raw < threshold).astype(int)
    else:
        out["TARGET_BINARY"] = (y_raw >= threshold).astype(int)

    return out

if TARGET_COLUMN_STD not in df_work.columns:
    raise KeyError(f"{TARGET_COLUMN_STD} tidak ditemukan setelah cleaning")

df_model = create_binary_target(df_work, TARGET_COLUMN_STD, ADHERENCE_THRESHOLD, POSITIVE_IF_LOWER)

print("Distribusi target asli ADHERENCE")
display(pd.to_numeric(df_model[TARGET_COLUMN_STD], errors="coerce").value_counts().sort_index().to_frame("count"))

print("Distribusi target biner")
bin_vc = df_model["TARGET_BINARY"].value_counts().sort_index()
display(bin_vc.to_frame("count"))

n0 = int(bin_vc.get(0, 0))
n1 = int(bin_vc.get(1, 0))
minority = min(n0, n1)
majority = max(n0, n1)
imb_ratio = majority / max(minority, 1)
minority_pct = 100 * minority / max((n0 + n1), 1)

print(f"Minority count: {minority}")
print(f"Minority percentage: {minority_pct:.2f}%")
print(f"Imbalance ratio (majority/minority): {imb_ratio:.3f}")


In [ ]:
# Pisahkan fitur, target, dan group (pasien)
groups = df_model[GROUP_COLUMN_STD].astype("string").fillna("UNKNOWN_MEMBER").copy()

X = df_model.drop(columns=[TARGET_COLUMN_STD, "TARGET_BINARY", GROUP_COLUMN_STD], errors="ignore").copy()
y = df_model["TARGET_BINARY"].astype(int).copy()

num_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Jumlah pasien unik:", groups.nunique())
print("Jumlah fitur numerik:", len(num_cols))
print("Jumlah fitur kategorikal:", len(cat_cols))

print("Fitur numerik (sample 30):")
print(num_cols[:30])
print("Fitur kategorikal (sample 30):")
print(cat_cols[:30])


In [ ]:
# Train-test split holdout berbasis pasien (group-aware)
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_val = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train_val = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_train_val = groups.iloc[train_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

print("Train_val shape:", X_train_val.shape)
print("Test shape     :", X_test.shape)
print("Patients train_val:", groups_train_val.nunique())
print("Patients test     :", groups_test.nunique())
print("Patient overlap   :", len(set(groups_train_val.unique()) & set(groups_test.unique())))

print("Distribusi kelas train_val")
print(y_train_val.value_counts(normalize=True).sort_index())
print("Distribusi kelas test")
print(y_test.value_counts(normalize=True).sort_index())


In [ ]:
# Preprocessing pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor_template = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols),
])

# Demo fit hanya untuk melihat jumlah fitur setelah encoding (fit di train_val saja)
preprocessor_demo = preprocessor_template
X_train_val_proc_demo = preprocessor_demo.fit_transform(X_train_val)
feature_names_demo = preprocessor_demo.get_feature_names_out()

print("Jumlah fitur sebelum encoding:", X_train_val.shape[1])
print("Jumlah fitur sesudah encoding:", X_train_val_proc_demo.shape[1])
print("Contoh nama fitur hasil encoding:")
print(feature_names_demo[:20])


In [ ]:
# Definisi metrik evaluasi

def _specificity_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0


def evaluate_model(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    pr_auc = auc(recall, precision)

    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": _specificity_from_cm(cm),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pr_auc": pr_auc,
        "confusion_matrix": cm,
    }


In [ ]:
# Definisi imbalance handling

def apply_imbalance_handling(X_train, y_train, method: str, random_state: int = 42):
    y_train = np.asarray(y_train)

    def _print_dist(tag, arr):
        vals, cnts = np.unique(arr, return_counts=True)
        msg = {int(v): int(c) for v, c in zip(vals, cnts)}
        print(f"{tag}: {msg}")

    _print_dist("Before", y_train)

    if method == "none":
        _print_dist("After", y_train)
        return X_train, y_train, None

    if method == "random_oversampling":
        sampler = RandomOverSampler(random_state=random_state)
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        _print_dist("After", y_res)
        return X_res, y_res, None

    if method == "smote":
        sampler = SMOTE(random_state=random_state)
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        _print_dist("After", y_res)
        return X_res, y_res, None

    if method == "random_undersampling":
        sampler = RandomUnderSampler(random_state=random_state)
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        _print_dist("After", y_res)
        return X_res, y_res, None

    if method == "class_weight":
        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
        class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
        print("After: unchanged (class_weight mode)")
        print("class_weight:", class_weight)
        return X_train, y_train, class_weight

    raise ValueError(f"Unknown method: {method}")


In [ ]:
# Definisi model deep learning

def build_model(input_dim: int, model_type: str = "mlp_baseline", learning_rate: float = 1e-3):
    if model_type == "mlp_baseline":
        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(64, activation="relu"),
            layers.Dropout(0.30),
            layers.Dense(1, activation="sigmoid"),
        ])
    elif model_type == "deep_dense":
        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(128, activation="relu"),
            layers.BatchNormalization(),
            layers.Dropout(0.35),
            layers.Dense(64, activation="relu"),
            layers.Dropout(0.25),
            layers.Dense(1, activation="sigmoid"),
        ])
    else:
        raise ValueError("model_type harus 'mlp_baseline' atau 'deep_dense'")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(curve="PR", name="pr_auc"), keras.metrics.Recall(name="recall")],
    )
    return model


## Setup 5-Fold Cross-Validation

Cross-validation dilakukan **hanya pada train_val set (80%)**.
- `test set` final (20%) disimpan terpisah dan tidak digunakan selama pemilihan model.
- Pada setiap fold:
  1. preprocessing di-fit ulang hanya pada train fold
  2. imbalance handling diterapkan hanya pada train fold
  3. validasi dilakukan pada val fold tanpa resampling

Pendekatan ini mencegah data leakage lintas fold.


In [ ]:
cv_splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("StratifiedGroupKFold ready:", cv_splitter)


In [ ]:
# Jalankan eksperimen K-Fold (group-aware by patient)
imbalance_methods = ["none", "random_oversampling", "smote", "random_undersampling", "class_weight"]
model_types = ["mlp_baseline", "deep_dense"]

EPOCHS = 20
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
PATIENCE = 4

cv_rows = []

X_train_val_reset = X_train_val.reset_index(drop=True)
y_train_val_reset = y_train_val.reset_index(drop=True)
groups_train_val_reset = groups_train_val.reset_index(drop=True)

for model_type in model_types:
    for imb_method in imbalance_methods:
        print(f"\n===== MODEL={model_type} | IMB={imb_method} =====")

        splits = cv_splitter.split(X_train_val_reset, y_train_val_reset, groups=groups_train_val_reset)
        for fold_idx, (tr_idx, va_idx) in enumerate(splits, start=1):
            print(f"Fold {fold_idx}/{N_SPLITS} ...")

            X_tr_df = X_train_val_reset.iloc[tr_idx].copy()
            y_tr = y_train_val_reset.iloc[tr_idx].to_numpy()
            X_va_df = X_train_val_reset.iloc[va_idx].copy()
            y_va = y_train_val_reset.iloc[va_idx].to_numpy()
            g_tr = groups_train_val_reset.iloc[tr_idx]
            g_va = groups_train_val_reset.iloc[va_idx]

            overlap = len(set(g_tr.unique()) & set(g_va.unique()))
            if overlap != 0:
                raise RuntimeError(f"Patient leakage terdeteksi pada fold {fold_idx}: overlap={overlap}")

            # Fit preprocessor hanya di train fold
            preprocessor_fold = preprocessor_template
            X_tr = preprocessor_fold.fit_transform(X_tr_df)
            X_va = preprocessor_fold.transform(X_va_df)

            # Imbalance handling hanya pada train fold
            try:
                X_tr_used, y_tr_used, cw = apply_imbalance_handling(
                    X_tr, y_tr, method=imb_method, random_state=RANDOM_STATE
                )
            except Exception as e:
                print(f"[WARN] Imbalance method failed on fold {fold_idx}: {e}")
                continue

            model = build_model(
                input_dim=X_tr.shape[1],
                model_type=model_type,
                learning_rate=LEARNING_RATE,
            )

            callbacks = [
                keras.callbacks.EarlyStopping(
                    monitor="val_pr_auc",
                    mode="max",
                    patience=PATIENCE,
                    restore_best_weights=True,
                )
            ]

            history = model.fit(
                X_tr_used,
                y_tr_used,
                validation_data=(X_va, y_va),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                verbose=0,
                callbacks=callbacks,
                class_weight=cw,
            )

            y_va_proba = model.predict(X_va, verbose=0).reshape(-1)
            y_va_pred = (y_va_proba >= 0.5).astype(int)

            m = evaluate_model(y_va, y_va_pred, y_va_proba)
            cv_rows.append(
                {
                    "fold": fold_idx,
                    "model_type": model_type,
                    "imbalance_method": imb_method,
                    "precision": m["precision"],
                    "recall": m["recall"],
                    "specificity": m["specificity"],
                    "f1": m["f1"],
                    "pr_auc": m["pr_auc"],
                    "epochs_ran": len(history.history.get("loss", [])),
                }
            )

cv_results = pd.DataFrame(cv_rows)
print("\nCV results shape:", cv_results.shape)
display(cv_results.head())


In [ ]:
# Ringkasan hasil CV
agg = (
    cv_results
    .groupby(["model_type", "imbalance_method"], as_index=False)
    .agg(
        mean_precision=("precision", "mean"),
        std_precision=("precision", "std"),
        mean_recall=("recall", "mean"),
        std_recall=("recall", "std"),
        mean_specificity=("specificity", "mean"),
        std_specificity=("specificity", "std"),
        mean_f1=("f1", "mean"),
        std_f1=("f1", "std"),
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
    )
    .sort_values(["mean_pr_auc", "mean_recall", "mean_f1"], ascending=False)
    .reset_index(drop=True)
)

display(agg.head(10))


## Visualisasi Hasil K-Fold

Grafik berikut membandingkan performa rata-rata antar metode imbalance dan arsitektur model.
Error bar menunjukkan deviasi standar antar fold.


In [ ]:
top_plot = agg.copy()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.barplot(data=top_plot, x="imbalance_method", y="mean_pr_auc", hue="model_type", errorbar=None)
for i, row in top_plot.iterrows():
    pass
plt.xticks(rotation=25)
plt.title("Mean PR-AUC by Imbalance Method")

plt.subplot(1, 2, 2)
sns.barplot(data=top_plot, x="imbalance_method", y="mean_recall", hue="model_type", errorbar=None)
plt.xticks(rotation=25)
plt.title("Mean Recall by Imbalance Method")

plt.tight_layout()
plt.show()

print("Interpretasi singkat:")
print("- Prioritaskan kombinasi dengan mean PR-AUC tinggi dan recall kelas minoritas baik.")
print("- Std yang besar menandakan performa kurang stabil antar fold.")


In [ ]:
# Pilih model terbaik
if agg.empty:
    raise RuntimeError("Agg result kosong. Pastikan CV berhasil dijalankan.")

best_row = agg.sort_values(["mean_pr_auc", "mean_recall", "mean_f1"], ascending=False).iloc[0]

best_model_type = best_row["model_type"]
best_imbalance_method = best_row["imbalance_method"]

print("Best model type:", best_model_type)
print("Best imbalance method:", best_imbalance_method)
display(best_row.to_frame("value"))


In [ ]:
# Training final model pada seluruh train_val (patient-group-aware split sudah dilakukan di tahap holdout)
final_preprocessor = preprocessor_template
X_train_val_proc = final_preprocessor.fit_transform(X_train_val)
X_test_proc = final_preprocessor.transform(X_test)

X_train_final, y_train_final, final_class_weight = apply_imbalance_handling(
    X_train_val_proc,
    y_train_val.to_numpy(),
    method=best_imbalance_method,
    random_state=RANDOM_STATE,
)

final_model = build_model(
    input_dim=X_train_val_proc.shape[1],
    model_type=best_model_type,
    learning_rate=LEARNING_RATE,
)

final_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_pr_auc",
        mode="max",
        patience=PATIENCE,
        restore_best_weights=True,
    )
]

final_history = final_model.fit(
    X_train_final,
    y_train_final,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
    callbacks=final_callbacks,
    class_weight=final_class_weight,
)

feature_names_final = final_preprocessor.get_feature_names_out().tolist()

final_config = {
    "dataset_mode": DATASET_MODE,
    "max_rows": MAX_ROWS,
    "random_state": RANDOM_STATE,
    "n_splits": N_SPLITS,
    "test_size": TEST_SIZE,
    "adherence_threshold": ADHERENCE_THRESHOLD,
    "group_column": GROUP_COLUMN,
    "split_strategy": "GroupShuffleSplit by patient",
    "cv_strategy": "StratifiedGroupKFold by patient",
    "best_model_type": best_model_type,
    "best_imbalance_method": best_imbalance_method,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
}

print("Final model trained.")
print(final_config)


In [ ]:
# Evaluasi final pada test set
y_test_proba = final_model.predict(X_test_proc, verbose=0).reshape(-1)
y_test_pred = (y_test_proba >= 0.5).astype(int)

final_metrics = evaluate_model(y_test.to_numpy(), y_test_pred, y_test_proba)

print("Classification report (test)")
print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))

print("Final metrics")
for k in ["precision", "recall", "specificity", "f1", "pr_auc"]:
    print(f"- {k}: {final_metrics[k]:.4f}")

precision_arr, recall_arr, _ = precision_recall_curve(y_test, y_test_proba)

final_metrics_df = pd.DataFrame([{
    "precision": final_metrics["precision"],
    "recall": final_metrics["recall"],
    "specificity": final_metrics["specificity"],
    "f1": final_metrics["f1"],
    "pr_auc": final_metrics["pr_auc"],
}])
final_metrics_df.to_csv(METRICS_DIR / "final_test_metrics.csv", index=False)
print("Saved:", METRICS_DIR / "final_test_metrics.csv")


In [ ]:
# Confusion matrix plot
cm = final_metrics["confusion_matrix"]

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix (Test)")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks([0.5, 1.5], ["0=adherent", "1=non-adherent"])
plt.yticks([0.5, 1.5], ["0=adherent", "1=non-adherent"], rotation=0)
plt.tight_layout()

cm_path = FIGURES_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", cm_path)


In [ ]:
# Precision-Recall curve
plt.figure(figsize=(6, 5))
plt.plot(recall_arr, precision_arr, label=f"PR-AUC = {final_metrics['pr_auc']:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Test)")
plt.legend()
plt.tight_layout()

pr_path = FIGURES_DIR / "pr_curve.png"
plt.savefig(pr_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", pr_path)


## Persiapan SHAP

SHAP dijalankan setelah model terbaik selesai dipilih dan dievaluasi.
Untuk efisiensi komputasi, digunakan subset data test (mis. 200–500 sampel).
Nama fitur hasil encoding diambil dari preprocessor agar interpretasi tetap terbaca.


In [ ]:
# Persiapan data untuk SHAP
n_bg = min(200, X_train_val_proc.shape[0])
n_explain = min(300, X_test_proc.shape[0])

rng = np.random.default_rng(RANDOM_STATE)
bg_idx = rng.choice(X_train_val_proc.shape[0], size=n_bg, replace=False)
ex_idx = rng.choice(X_test_proc.shape[0], size=n_explain, replace=False)

X_background = X_train_val_proc[bg_idx]
X_explain = X_test_proc[ex_idx]
y_explain_true = y_test.iloc[ex_idx].to_numpy()
y_explain_proba = y_test_proba[ex_idx]
y_explain_pred = (y_explain_proba >= 0.5).astype(int)

feature_names = feature_names_final
print("Background shape:", X_background.shape)
print("Explain shape:", X_explain.shape)
print("Num features:", len(feature_names))


In [ ]:
# Jalankan SHAP

def model_predict(x):
    return final_model.predict(x, verbose=0).reshape(-1)

try:
    explainer = shap.DeepExplainer(final_model, X_background)
    shap_values_raw = explainer.shap_values(X_explain)
    print("Using DeepExplainer")
except Exception as e:
    print("DeepExplainer gagal, fallback ke KernelExplainer:", e)
    explainer = shap.KernelExplainer(model_predict, X_background)
    shap_values_raw = explainer.shap_values(X_explain, nsamples=100)

shap_values = np.array(shap_values_raw[0] if isinstance(shap_values_raw, list) else shap_values_raw)
if shap_values.ndim == 3 and shap_values.shape[-1] == 1:
    shap_values = shap_values[..., 0]

# summary plot
plt.figure()
shap.summary_plot(shap_values, X_explain, feature_names=feature_names, show=False)
summary_path = SHAP_DIR / "shap_summary.png"
plt.tight_layout()
plt.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.show()

# bar plot top-20
plt.figure()
shap.summary_plot(shap_values, X_explain, feature_names=feature_names, plot_type="bar", max_display=20, show=False)
bar_path = SHAP_DIR / "shap_bar_top20.png"
plt.tight_layout()
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved SHAP summary:", summary_path)
print("Saved SHAP bar:", bar_path)


## Interpretasi Global SHAP

Interpretasi global SHAP digunakan untuk melihat fitur mana yang paling berpengaruh terhadap output model.
Interpretasi ini membantu diskusi faktor yang terkait prediksi non-adherence, namun **tidak** boleh langsung dianggap kausal/klinis tanpa validasi medis lanjutan.


In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
global_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

top20_global = global_importance.head(20)
display(top20_global)

print("Interpretasi ringkas:")
print("- Fitur dengan mean |SHAP| besar paling sering mempengaruhi prediksi model.")
print("- Nilai SHAP positif cenderung mendorong probabilitas ke kelas non-adherent (1).")
print("- Nilai SHAP negatif cenderung mendorong probabilitas ke kelas adherent (0).")


In [ ]:
# Interpretasi lokal SHAP: TP, FP, FN
case_indices = {}

for i, (yt, yp) in enumerate(zip(y_test.to_numpy(), y_test_pred)):
    if yt == 1 and yp == 1 and "TP" not in case_indices:
        case_indices["TP"] = i
    if yt == 0 and yp == 1 and "FP" not in case_indices:
        case_indices["FP"] = i
    if yt == 1 and yp == 0 and "FN" not in case_indices:
        case_indices["FN"] = i
    if len(case_indices) == 3:
        break

print("Selected local cases:", case_indices)

local_rows = []
for case_name, idx in case_indices.items():
    x_row = X_test_proc[idx:idx+1]

    try:
        sv_raw = explainer.shap_values(x_row)
    except Exception:
        sv_raw = shap.KernelExplainer(model_predict, X_background).shap_values(x_row, nsamples=100)

    sv = np.array(sv_raw[0] if isinstance(sv_raw, list) else sv_raw).reshape(-1)
    top_idx = np.argsort(np.abs(sv))[::-1][:15]

    plt.figure(figsize=(8, 5))
    vals = sv[top_idx]
    labels = [feature_names[j] for j in top_idx]
    colors = ["#d95f02" if v > 0 else "#1b9e77" for v in vals]
    plt.barh(range(len(vals)), vals[::-1], color=colors[::-1])
    plt.yticks(range(len(vals)), labels[::-1])
    plt.axvline(0, color="black", linewidth=1)
    plt.title(f"Local SHAP - {case_name} (index={idx})")
    plt.xlabel("SHAP contribution")
    plt.tight_layout()

    local_path = SHAP_DIR / f"local_{case_name}.png"
    plt.savefig(local_path, dpi=150, bbox_inches="tight")
    plt.show()

    local_rows.append({
        "case": case_name,
        "test_index": int(idx),
        "y_true": int(y_test.iloc[idx]),
        "y_pred": int(y_test_pred[idx]),
        "y_proba": float(y_test_proba[idx]),
        "plot_path": str(local_path),
    })

local_explanations_df = pd.DataFrame(local_rows)
display(local_explanations_df)


## Kesimpulan Eksperimen

Isi ringkasan hasil akhir setelah semua cell dijalankan:

1. **Model terbaik**: gunakan variabel `best_model_type`.
2. **Metode imbalance terbaik**: gunakan variabel `best_imbalance_method`.
3. **Performa test set**: gunakan nilai `precision`, `recall`, `specificity`, `f1`, `pr_auc` dari `final_metrics`.
4. **Insight SHAP**: gunakan tabel `top20_global` dan local explanation (TP/FP/FN).
5. **Keterbatasan**: eksperimen ini dibatasi maksimal 10.000 baris untuk efisiensi.
6. **Saran lanjutan**: jalankan ulang pipeline pada full dataset jika sumber daya komputasi memungkinkan.


In [ ]:
# Simpan artefak
cv_results_path = METRICS_DIR / "cv_results.csv"
cv_summary_path = METRICS_DIR / "cv_summary.csv"
final_metrics_path = METRICS_DIR / "final_test_metrics.csv"
model_path = MODELS_DIR / "final_model.keras"
preproc_path = MODELS_DIR / "preprocessing_pipeline.joblib"
best_cfg_path = MODELS_DIR / "best_config.json"

cv_results.to_csv(cv_results_path, index=False)
agg.to_csv(cv_summary_path, index=False)
final_metrics_df.to_csv(final_metrics_path, index=False)

final_model.save(model_path)
joblib.dump(final_preprocessor, preproc_path)

with open(best_cfg_path, "w") as f:
    json.dump(final_config, f, indent=2)

print("Saved artifacts:")
print("-", cv_results_path)
print("-", cv_summary_path)
print("-", final_metrics_path)
print("-", model_path)
print("-", preproc_path)
print("-", best_cfg_path)

print("Catatan:")
print("- Hasil subset 10.000 data digunakan sebagai simulasi awal.")
print("- Untuk pelaporan akhir tesis, jalankan ulang pada full dataset jika komputasi memadai.")
